In [1]:
!pip install pandas numpy matplotlib statsmodels scikit-learn


In [2]:
r"C:\Users\aarya\Downloads\household_power_consumption.txt"

'C:\\Users\\aarya\\Downloads\\household_power_consumption.txt'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX


In [5]:
file_path = r"C:\Users\aarya\Downloads\household_power_consumption.txt"

df = pd.read_csv(
    file_path,
    sep=";",
    na_values="?",
    low_memory=False
)


In [14]:
# Combine Date + Time properly
df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True
)

df.set_index("datetime", inplace=True)

# Convert power to numeric
df["Global_active_power"] = pd.to_numeric(
    df["Global_active_power"],
    errors="coerce"
)

df = df.dropna(subset=["Global_active_power"])


In [15]:
df["occupancy"] = (df["Global_active_power"] > 
                   df["Global_active_power"].median()).astype(int)


In [20]:
data = df[["Global_active_power", "occupancy"]].resample("h").mean()

data = data.interpolate()

data.rename(columns={"Global_active_power": "power"}, inplace=True)


In [1]:
train_size = int(len(data) * 0.8)
train = data.iloc[:train_size]
test = data.iloc[train_size:]
train = train.asfreq("H")
test = test.asfreq("H")


NameError: name 'data' is not defined

In [66]:
from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(train["power"], order=(2,1,2))
arima_fit = arima_model.fit()

print(arima_fit.summary())


C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


                               SARIMAX Results                                
Dep. Variable:                  power   No. Observations:                27334
Model:                 ARIMA(2, 1, 2)   Log Likelihood              -26355.626
Date:                Thu, 22 Jan 2026   AIC                          52721.251
Time:                        11:26:17   BIC                          52762.331
Sample:                             0   HQIC                         52734.491
                              - 27334                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          1.3929      0.022     64.670      0.000       1.351       1.435
ar.L2         -0.5486      0.014    -39.877      0.000      -0.576      -0.522
ma.L1         -1.6382      0.023    -71.572      0.0

In [53]:
data.index = pd.to_datetime(data.index)
data = data.asfreq("h")
data = data.dropna()


In [56]:
train_size = int(len(data) * 0.8)

train = data.iloc[:train_size].copy()
test = data.iloc[train_size:].copy()


In [64]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    endog=train["power"],
    exog=train["occupancy"],
    order=(1,1,1),
    enforce_stationarity=False,
    enforce_invertibility=False
)

fit = model.fit(disp=False)


C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [60]:
forecast = fit.get_forecast(
    steps=len(test),
    exog=test["occupancy"]
)

forecast_mean = forecast.predicted_mean
conf_int = forecast.conf_int()


C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\aarya\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [62]:
print(type(forecast_mean.index))


<class 'pandas.core.indexes.range.RangeIndex'>
